# CNN and Convolution Mathematics — Exercises

10 hands-on exercises covering the core mathematics of convolutional neural networks.
Each exercise has a **Task** section (attempt it yourself) and a **Solution** section.

**Difficulty:** ★☆☆ Easy  ★★☆ Medium  ★★★ Hard


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.linalg import toeplitz
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print("Setup complete ✓")


---
## Exercise 1 ★☆☆ — Output Size Formula

For a 2-D convolutional layer with:
- Input: $H_{in} \times W_{in}$
- Kernel: $k \times k$
- Padding: $p$
- Stride: $s$
- Dilation: $d$

The output size is:
$$
H_{out} = \left\lfloor \frac{H_{in} + 2p - d(k-1) - 1}{s} \right\rfloor + 1
$$

**Task:** Implement `output_size(H, W, k, p=0, s=1, d=1)` and verify against PyTorch-style
formulas for the following configs. Also derive: what padding $p$ is needed to keep
$H_{out} = H_{in}$ when $s=1$?


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────
    # Implement output_size and fill the table

    def output_size(H, W, k, p=0, s=1, d=1):
        """Compute (H_out, W_out) for a 2-D conv layer."""
        pass  # TODO

    configs = [
        # (H, W, k, p, s, d)
        (224, 224, 3, 0, 1, 1),
        (224, 224, 3, 1, 1, 1),
        (224, 224, 3, 1, 2, 1),
        (224, 224, 5, 2, 1, 1),
        (56, 56, 3, 1, 1, 2),
        (56, 56, 3, 2, 1, 2),
    ]
    for cfg in configs:
        H, W, k, p, s, d = cfg
        out = output_size(H, W, k, p, s, d)
        print(f"H={H}, k={k}, p={p}, s={s}, d={d} → {out}")


### Solution 1

In [ ]:
def output_size(H, W, k, p=0, s=1, d=1):
    H_out = (H + 2*p - d*(k-1) - 1) // s + 1
    W_out = (W + 2*p - d*(k-1) - 1) // s + 1
    return H_out, W_out

configs = [
    (224, 224, 3, 0, 1, 1),
    (224, 224, 3, 1, 1, 1),
    (224, 224, 3, 1, 2, 1),
    (224, 224, 5, 2, 1, 1),
    (56,  56,  3, 1, 1, 2),
    (56,  56,  3, 2, 1, 2),
]

print(f"{'H':>4} {'k':>2} {'p':>2} {'s':>2} {'d':>2} | {'H_out':>6} {'W_out':>6} | {'HW preserved?':>14}")
print("-"*55)
for cfg in configs:
    H, W, k, p, s, d = cfg
    Ho, Wo = output_size(H, W, k, p, s, d)
    preserved = "✓" if Ho == H and Wo == W else ""
    print(f"{H:4d} {k:2d} {p:2d} {s:2d} {d:2d} | {Ho:6d} {Wo:6d} | {preserved:>14}")

print()
print("Same-size condition (s=1, d=1): p = (k-1)/2  (requires odd k)")
print("Examples: k=3 → p=1,  k=5 → p=2,  k=7 → p=3")


---
## Exercise 2 ★☆☆ — Parameter Counting

**Task:** Compute the total trainable parameter count for the following architectures.
Include bias terms where they exist.

| Layer | Spec |
|---|---|
| Conv2d | $C_{in}=3$, $C_{out}=64$, $k=3$, bias=True |
| Conv2d | $C_{in}=64$, $C_{out}=128$, $k=3$, bias=True |
| BatchNorm2d | 128 features |
| Linear | in=4096, out=1000 |
| Depthwise | $C=64$, $k=3$, bias=False |
| Pointwise | $C_{in}=64$, $C_{out}=128$, bias=False |

Also: compare total params of `[Conv(3→64,3×3), Conv(64→64,3×3)]` vs
one equivalent `FC(H×W×3, H×W×64)` for $H=W=32$.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def conv_params(C_in, C_out, k, bias=True):
        pass  # TODO

    def bn_params(C):
        pass  # TODO (gamma, beta — both learnable)

    def fc_params(in_feat, out_feat, bias=True):
        pass  # TODO

    # Fill in param counts
    layers = [
        ("Conv(3→64,3×3)", conv_params(3, 64, 3)),
        ("Conv(64→128,3×3)", conv_params(64, 128, 3)),
        ("BN(128)", bn_params(128)),
        ("Linear(4096→1000)", fc_params(4096, 1000)),
        ("DW(64,3×3)", conv_params(1, 64, 3, bias=False)),  # one kernel per channel
        ("PW(64→128)", conv_params(64, 128, 1, bias=False)),
    ]
    for name, p in layers:
        print(f"{name:30s}: {p:>10,d}")


### Solution 2

In [ ]:
def conv_params(C_in, C_out, k, bias=True):
    """Standard conv: kernel params + optional bias."""
    return C_in * C_out * k * k + (C_out if bias else 0)

def dw_params(C, k, bias=False):
    """Depthwise: one k×k kernel per channel."""
    return C * k * k + (C if bias else 0)

def bn_params(C):
    """BatchNorm: gamma and beta, each of size C."""
    return 2 * C

def fc_params(in_feat, out_feat, bias=True):
    return in_feat * out_feat + (out_feat if bias else 0)

layers = [
    ("Conv(3→64,3×3,bias)",   conv_params(3, 64, 3, bias=True)),
    ("Conv(64→128,3×3,bias)", conv_params(64, 128, 3, bias=True)),
    ("BN(128)",               bn_params(128)),
    ("Linear(4096→1000)",     fc_params(4096, 1000)),
    ("DW(64,3×3)",            dw_params(64, 3)),
    ("PW(64→128)",            conv_params(64, 128, 1, bias=False)),
]
total = 0
print(f"{'Layer':<28} {'Params':>10}")
print("-"*40)
for name, p in layers:
    total += p
    print(f"{name:<28} {p:>10,d}")
print("-"*40)
print(f"{'TOTAL':<28} {total:>10,d}")

# CNN vs FC comparison (32×32 image)
H = W = 32
cnn_params = conv_params(3, 64, 3) + conv_params(64, 64, 3)
fc_equiv   = fc_params(H*W*3, H*W*64)
print(f"\nCNN [Conv(3→64,3×3) + Conv(64→64,3×3)]: {cnn_params:>10,d} params")
print(f"Equivalent FC (H×W×3 → H×W×64):          {fc_equiv:>10,d} params")
print(f"Reduction ratio: {fc_equiv/cnn_params:.0f}x fewer params in CNN")


---
## Exercise 3 ★★☆ — Implement 2-D Convolution with Full Support

**Task:** Implement `conv2d(X, K, stride=1, padding=0, dilation=1)` that handles
all four parameters correctly.  Validate by matching NumPy's `np.convolve` for 1-D
and by verifying the output-size formula from Exercise 1.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def conv2d(X, K, stride=1, padding=0, dilation=1):
        """
        2-D cross-correlation with stride, padding, and dilation.
        X: (H, W) input
        K: (kH, kW) kernel
        Returns: (H_out, W_out) output
        """
        pass  # TODO

    # Test cases
    X_test = np.arange(25, dtype=float).reshape(5, 5)
    K_test = np.array([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=float)  # Sobel
    print("No padding, stride 1:", conv2d(X_test, K_test).shape)          # expect (3,3)
    print("Padding=1,  stride 1:", conv2d(X_test, K_test, padding=1).shape)  # expect (5,5)
    print("Padding=1,  stride 2:", conv2d(X_test, K_test, padding=1, stride=2).shape)  # (3,3)
    print("Dilation=2, pad=2:  ", conv2d(X_test, K_test, padding=2, dilation=2).shape)  # (5,5)


### Solution 3

In [ ]:
def conv2d(X, K, stride=1, padding=0, dilation=1):
    if padding > 0:
        X = np.pad(X, padding)
    H, W   = X.shape
    kH, kW = K.shape
    kH_eff = kH + (kH-1)*(dilation-1)
    kW_eff = kW + (kW-1)*(dilation-1)
    oH = (H - kH_eff) // stride + 1
    oW = (W - kW_eff) // stride + 1
    out = np.zeros((oH, oW))
    for i in range(oH):
        for j in range(oW):
            for m in range(kH):
                for n in range(kW):
                    out[i, j] += X[i*stride + m*dilation, j*stride + n*dilation] * K[m, n]
    return out

# Verification
X_test = np.arange(25, dtype=float).reshape(5, 5)
K_test = np.array([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=float)

tests = [
    ("No pad, s=1",      dict()),
    ("pad=1, s=1",       dict(padding=1)),
    ("pad=1, s=2",       dict(padding=1, stride=2)),
    ("pad=2, dil=2",     dict(padding=2, dilation=2)),
    ("pad=1, s=1, d=2",  dict(padding=1, dilation=2)),
]
for name, kwargs in tests:
    out = conv2d(X_test, K_test, **kwargs)
    kH, kW = K_test.shape
    d = kwargs.get('dilation', 1)
    p = kwargs.get('padding', 0)
    s = kwargs.get('stride', 1)
    Ho_expected = (5 + 2*p - d*(kH-1) - 1)//s + 1
    status = "✓" if out.shape[0] == Ho_expected else f"✗ (expected {Ho_expected})"
    print(f"{name:<20}: shape={out.shape}  {status}")

# Edge: random vs reference
import scipy.ndimage as ndi
X_r = np.random.randn(8, 8)
K_r = np.random.randn(3, 3)
our = conv2d(X_r, K_r)
ref = ndi.correlate(X_r, K_r, mode='constant')[1:-1, 1:-1]  # valid region
print(f"\nRandom test vs scipy: max_diff={np.max(np.abs(our-ref)):.2e}")


---
## Exercise 4 ★☆☆ — Sobel Edge Detection

**Task:** Apply Sobel edge detection to a synthetic image.
1. Create a $64\times64$ image with geometric shapes.
2. Apply horizontal ($G_x$) and vertical ($G_y$) Sobel kernels.
3. Compute gradient magnitude $G = \sqrt{G_x^2 + G_y^2}$ and direction $\theta = \arctan(G_y/G_x)$.
4. Visualise all four results in a $2\times2$ grid.

Also show why Sobel is *separable* and factor it into $\mathbf{k}_v \otimes \mathbf{k}_h^T$.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    # Create synthetic image
    def make_test_image(size=64):
        img = np.zeros((size, size))
        # Square
        img[15:35, 15:35] = 1.0
        # Circle
        cy, cx, r = 48, 48, 10
        Y, X = np.ogrid[:size, :size]
        mask = (X-cx)**2 + (Y-cy)**2 <= r**2
        img[mask] = 0.7
        # Triangle (ramp)
        for i in range(20):
            img[5+i, 50:50+i] = 0.5
        return img

    img = make_test_image()
    # TODO: apply Sobel, compute magnitude and direction, visualise


### Solution 4

In [ ]:
def make_test_image(size=64):
    img = np.zeros((size, size))
    img[15:35, 15:35] = 1.0
    cy, cx, r = 48, 48, 10
    Y, X = np.ogrid[:size, :size]
    img[(X-cx)**2 + (Y-cy)**2 <= r**2] = 0.7
    for i in range(20):
        img[5+i, 50:50+i] = 0.5
    return img

img = make_test_image()

# Sobel kernels
Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
Ky = np.array([[-1,-2,-1], [ 0, 0, 0], [ 1, 2, 1]], dtype=float)

Gx   = conv2d(img, Kx, padding=1)
Gy   = conv2d(img, Ky, padding=1)
G    = np.sqrt(Gx**2 + Gy**2)
theta = np.arctan2(Gy, Gx) * 180 / np.pi

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
imgs_data = [img, Gx, Gy, G, theta, np.abs(theta)]
titles    = ['Original', 'Gx (horizontal)', 'Gy (vertical)',
             'Gradient Magnitude', 'Direction (°)', '|Direction| (°)']
cmaps     = ['gray', 'RdBu', 'RdBu', 'hot', 'hsv', 'viridis']
for ax, data, title, cmap in zip(axes.flat, imgs_data, titles, cmaps):
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Sobel Edge Detection', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Separability: Kx = [1,2,1]^T ⊗ [-1,0,1]
kv = np.array([1., 2., 1.]).reshape(3, 1)
kh = np.array([-1., 0., 1.]).reshape(1, 3)
Kx_sep = kv @ kh

print("Kx is separable:", np.allclose(Kx, Kx_sep))
print("Kx  =\n", Kx.astype(int))
print("kv⊗kh =\n", Kx_sep.astype(int))
print("\nSeparable conv advantage: O(k) instead of O(k²) per pixel")
print(f"For k=3: 6 multiplications per pixel instead of 9 (save {1-6/9:.0%})")


---
## Exercise 5 ★★☆ — Backpropagation Gradient Check

**Task:** Implement the backward pass of a 2-D convolutional layer and verify it
against numerical gradients.

Given forward: $Y = X \star K$, derive:

1. $\nabla_X \mathcal{L}$ — gradient w.r.t. input (full conv with flipped $K$)
2. $\nabla_K \mathcal{L}$ — gradient w.r.t. kernel (cross-corr of $X$ with $dY$)

Use the finite-difference approximation with $\epsilon=10^{-5}$ to verify both.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def conv2d_backward(X, K, dY):
        """
        Returns dX (same shape as X) and dK (same shape as K).
        Assume VALID padding (no padding in forward).
        """
        dX = None  # TODO
        dK = None  # TODO
        return dX, dK

    # Test
    X_t = np.random.randn(6, 6)
    K_t = np.random.randn(3, 3)
    Y_t = conv2d(X_t, K_t)
    dY_t = np.random.randn(*Y_t.shape)

    dX, dK = conv2d_backward(X_t, K_t, dY_t)
    print("dX shape:", dX.shape if dX is not None else "None")
    print("dK shape:", dK.shape if dK is not None else "None")


### Solution 5

In [ ]:
def conv2d_backward(X, K, dY):
    H, W   = X.shape
    kH, kW = K.shape
    oH, oW = dY.shape

    # ── dX: full conv of dY with flipped K ────────────────────────────────────
    # "Full" = pad dY by (k-1) on each side, then cross-correlate with flipped K
    K_flip    = K[::-1, ::-1]
    dY_padded = np.pad(dY, ((kH-1, kH-1), (kW-1, kW-1)))
    dX = np.zeros_like(X)
    for i in range(H):
        for j in range(W):
            dX[i, j] = np.sum(dY_padded[i:i+kH, j:j+kW] * K_flip)

    # ── dK: cross-corr of X with dY ───────────────────────────────────────────
    dK = np.zeros_like(K)
    for m in range(kH):
        for n in range(kW):
            dK[m, n] = np.sum(X[m:m+oH, n:n+oW] * dY)

    return dX, dK


def numerical_grad(func, param, eps=1e-5):
    """Finite-difference gradient of scalar-valued func w.r.t. param."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = param[idx]
        param[idx] = orig + eps; f_plus  = func(param)
        param[idx] = orig - eps; f_minus = func(param)
        param[idx] = orig
        grad[idx]  = (f_plus - f_minus) / (2 * eps)
        it.iternext()
    return grad

np.random.seed(0)
X_t = np.random.randn(6, 6)
K_t = np.random.randn(3, 3)
dY  = np.random.randn(4, 4)   # valid output shape

dX_analytic, dK_analytic = conv2d_backward(X_t, K_t, dY)

# Loss = sum(Y * dY) — so dL/dY = dY
loss_fn = lambda X: np.sum(conv2d(X, K_t) * dY)
loss_kn = lambda K: np.sum(conv2d(X_t, K) * dY)

dX_num = numerical_grad(loss_fn, X_t.copy())
dK_num = numerical_grad(loss_kn, K_t.copy())

print("dX gradient check:", "PASS ✓" if np.allclose(dX_analytic, dX_num, atol=1e-5) else "FAIL ✗")
print(f"  max|dX_analytic - dX_numeric| = {np.max(np.abs(dX_analytic-dX_num)):.2e}")
print("dK gradient check:", "PASS ✓" if np.allclose(dK_analytic, dK_num, atol=1e-5) else "FAIL ✗")
print(f"  max|dK_analytic - dK_numeric| = {np.max(np.abs(dK_analytic-dK_num)):.2e}")

# Visualise gradient structure
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, arr, title in zip(axes[0], [X_t, K_t, dY],
                          ['Input X', 'Kernel K', 'Upstream grad dY']):
    im = ax.imshow(arr, cmap='RdBu'); ax.set_title(title); plt.colorbar(im, ax=ax)
for ax, arr, title in zip(axes[1], [dX_analytic, dK_analytic, dX_analytic-dX_num],
                          ['dX (analytic)', 'dK (analytic)', 'dX error (should≈0)']):
    im = ax.imshow(arr, cmap='RdBu'); ax.set_title(title); plt.colorbar(im, ax=ax)
plt.suptitle('Conv Backprop: Analytical vs Numerical Gradients', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## Exercise 6 ★★★ — BatchNorm Backward Pass (Full Derivation)

**Task:** Derive and implement the BatchNorm backward pass analytically.

Starting from the forward pass $\hat{x}_i = (x_i - \mu)/\hat{\sigma}$, $y_i = \gamma\hat{x}_i + \beta$:

1. Derive $\partial\mathcal{L}/\partial\gamma$ and $\partial\mathcal{L}/\partial\beta$
2. Derive $\partial\mathcal{L}/\partial x_i$ (the tricky part — $\mu$ and $\sigma$ depend on all $x_j$)
3. Implement and verify with numerical gradients
4. Visualise the conditioning effect on gradient magnitudes


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def batchnorm_forward(x, gamma, beta, eps=1e-5):
        """x: (N, D). Returns (y, cache) for backward."""
        mu  = x.mean(axis=0)
        var = x.var(axis=0)
        x_hat = (x - mu) / np.sqrt(var + eps)
        y = gamma * x_hat + beta
        cache = (x, mu, var, x_hat, gamma, eps)
        return y, cache

    def batchnorm_backward(dout, cache):
        """Returns dx, dgamma, dbeta."""
        x, mu, var, x_hat, gamma, eps = cache
        N = x.shape[0]
        sigma = np.sqrt(var + eps)

        dgamma = None  # TODO
        dbeta  = None  # TODO
        dx     = None  # TODO
        return dx, dgamma, dbeta

    # Test
    x_t = np.random.randn(16, 8)
    g_t = np.random.randn(8); b_t = np.random.randn(8)
    y_t, cache_t = batchnorm_forward(x_t, g_t, b_t)
    dout_t = np.random.randn(*y_t.shape)
    dx, dg, db = batchnorm_backward(dout_t, cache_t)
    print("dx:", dx.shape if dx is not None else None)


### Solution 6

In [ ]:
def batchnorm_forward(x, gamma, beta, eps=1e-5):
    mu    = x.mean(axis=0)
    var   = x.var(axis=0)
    x_hat = (x - mu) / np.sqrt(var + eps)
    y     = gamma * x_hat + beta
    return y, (x, mu, var, x_hat, gamma, eps)

def batchnorm_backward(dout, cache):
    """
    Full analytical backward pass for BatchNorm.

    Derivation sketch:
      L = f(y) = f(γ x̂ + β)
      ∂L/∂β    = Σ_i ∂L/∂y_i            (β is just added)
      ∂L/∂γ    = Σ_i ∂L/∂y_i · x̂_i     (γ multiplies x̂)
      ∂L/∂x̂_i = ∂L/∂y_i · γ

    Then chain through x̂_i = (x_i - μ)/σ where μ = mean(x), σ² = var(x):
      ∂L/∂x_i = (γ/Nσ)[N·∂L/∂y_i·γ⁻¹·γ - Σ_j ∂L/∂y_j - x̂_i·Σ_j ∂L/∂y_j·x̂_j]
              = (1/Nσ)[N·dx̂_i - Σ_j dx̂_j - x̂_i·Σ_j dx̂_j·x̂_j]
    """
    x, mu, var, x_hat, gamma, eps = cache
    N     = x.shape[0]
    sigma = np.sqrt(var + eps)

    dbeta  = dout.sum(axis=0)
    dgamma = (dout * x_hat).sum(axis=0)

    dx_hat = dout * gamma
    dx = (1.0 / (N * sigma)) * (
        N * dx_hat
        - dx_hat.sum(axis=0)
        - x_hat * (dx_hat * x_hat).sum(axis=0)
    )
    return dx, dgamma, dbeta

# ── Numerical gradient check ────────────────────────────────────────────────
np.random.seed(7)
x_t   = np.random.randn(8, 5)
g_t   = np.abs(np.random.randn(5)) + 0.5
b_t   = np.random.randn(5)
dout  = np.random.randn(8, 5)

y_t, cache = batchnorm_forward(x_t, g_t, b_t)
dx, dg, db = batchnorm_backward(dout, cache)

eps = 1e-5

# Numerical dx
dx_num = np.zeros_like(x_t)
for i in range(x_t.shape[0]):
    for j in range(x_t.shape[1]):
        xp = x_t.copy(); xp[i,j] += eps; yp,_ = batchnorm_forward(xp, g_t, b_t)
        xm = x_t.copy(); xm[i,j] -= eps; ym,_ = batchnorm_forward(xm, g_t, b_t)
        dx_num[i,j] = np.sum(dout*(yp-ym))/(2*eps)

# Numerical dgamma
dg_num = np.zeros_like(g_t)
for j in range(g_t.shape[0]):
    gp = g_t.copy(); gp[j] += eps; yp,_ = batchnorm_forward(x_t, gp, b_t)
    gm = g_t.copy(); gm[j] -= eps; ym,_ = batchnorm_forward(x_t, gm, b_t)
    dg_num[j] = np.sum(dout*(yp-ym))/(2*eps)

# Numerical dbeta
db_num = np.zeros_like(b_t)
for j in range(b_t.shape[0]):
    bp = b_t.copy(); bp[j] += eps; yp,_ = batchnorm_forward(x_t, g_t, bp)
    bm = b_t.copy(); bm[j] -= eps; ym,_ = batchnorm_forward(x_t, g_t, bm)
    db_num[j] = np.sum(dout*(yp-ym))/(2*eps)

for name, a, n in [('dx', dx, dx_num), ('dgamma', dg, dg_num), ('dbeta', db, db_num)]:
    ok = np.allclose(a, n, atol=1e-5)
    print(f"{name:8s}: {'PASS ✓' if ok else 'FAIL ✗'}  max_err={np.max(np.abs(a-n)):.2e}")

# Conditioning visualisation
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x_bad = np.random.randn(100, 4) * np.array([10, 0.1, 5, 2]) + np.array([5,-3,0,8])
y_good, _ = batchnorm_forward(x_bad, np.ones(4), np.zeros(4))
for i in range(4):
    axes[0].hist(x_bad[:, i], bins=20, alpha=0.6, label=f'feat {i}')
    axes[1].hist(y_good[:, i], bins=20, alpha=0.6, label=f'feat {i}')
axes[0].set_title('Before BN (wildly different scales)'); axes[0].legend(fontsize=9)
axes[1].set_title('After BN (unit variance each)'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()


---
## Exercise 7 ★★☆ — Receptive Field Computation

The **effective receptive field** of layer $l$ is:
$$
r_l = r_{l-1} + (k_l - 1) \cdot \prod_{i=1}^{l-1} s_i
$$

**Task:**
1. Implement `compute_rf(layers)` where `layers` is a list of `(k, s)` tuples.
2. Compute the RF for VGG-16, ResNet-50 (first 4 stages), and a dilated network.
3. Show that two stacked $3\times3$ convs have the same RF as one $5\times5$ conv but
   use fewer parameters.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def compute_rf(layers):
        """
        layers: list of (kernel_size, stride) tuples.
        Returns (rf_sizes, jump_sizes) — RF and feature stride at each layer.
        """
        pass  # TODO

    # Example: simple 3-layer network
    simple = [(3,1), (3,1), (3,1)]
    rfs, jumps = compute_rf(simple)
    print("Simple network RF:", rfs)


### Solution 7

In [ ]:
def compute_rf(layers):
    """
    Returns (rf_list, jump_list).
    rf_list[i]   = receptive field size after layer i (1-indexed)
    jump_list[i] = jump (feature stride) after layer i
    """
    rf   = 1    # input pixel RF is 1
    jump = 1    # jump in input coords per output coord
    rf_list, jump_list = [1], [1]   # include input layer
    for k, s in layers:
        rf   = rf + (k - 1) * jump
        jump = jump * s
        rf_list.append(rf)
        jump_list.append(jump)
    return rf_list, jump_list

# Three stacked 3×3 vs one 7×7
two_3x3 = [(3,1), (3,1)]
one_5x5 = [(5,1)]
three_3x3 = [(3,1),(3,1),(3,1)]
one_7x7 = [(7,1)]

print("2× (3×3): RF =", compute_rf(two_3x3)[0][-1], " params per feature =", 2*9)
print("1× (5×5): RF =", compute_rf(one_5x5)[0][-1], " params per feature =", 25)
print("3× (3×3): RF =", compute_rf(three_3x3)[0][-1], " params per feature =", 3*9)
print("1× (7×7): RF =", compute_rf(one_7x7)[0][-1], " params per feature =", 49)

# VGG-16 convolution layers
vgg16_layers = [
    (3,1),(3,1),(2,2),          # block 1 (pool at end)
    (3,1),(3,1),(2,2),          # block 2
    (3,1),(3,1),(3,1),(2,2),    # block 3
    (3,1),(3,1),(3,1),(2,2),    # block 4
    (3,1),(3,1),(3,1),(2,2),    # block 5
]
vgg_rf, vgg_j = compute_rf(vgg16_layers)
print(f"\nVGG-16 final RF: {vgg_rf[-1]}  (on 224×224 input)")
print(f"VGG-16 jump:     {vgg_j[-1]}")

# ResNet-50 stem + first 4 stages (simplified)
resnet_layers = [
    (7,2),(3,2),       # stem: 7×7 conv + 3×3 maxpool
    (1,1),(3,1),(1,1), # stage1: bottleneck ×3 (only show one)
    (1,1),(3,2),(1,1), # stage2: bottleneck ×4
    (1,1),(3,2),(1,1), # stage3: bottleneck ×6
    (1,1),(3,2),(1,1), # stage4: bottleneck ×3
]
rn_rf, rn_j = compute_rf(resnet_layers)
print(f"\nResNet-50 approx final RF: {rn_rf[-1]}  jump: {rn_j[-1]}")

# Dilated network (no stride, exponentially growing RF)
dilated = []
for d in [1,2,4,8,16]:
    # 3×3 with dilation d has effective size d*(3-1)+1 = 2d+1
    k_eff = 2*d + 1
    dilated.append((k_eff, 1))
dil_rf, _ = compute_rf(dilated)
print(f"\nDilated network (d=1,2,4,8,16) RF after each layer: {dil_rf[1:]}")

# Plot RF growth
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(vgg_rf)), vgg_rf, 'C0o-', lw=2, label='VGG-16')
ax.plot(range(len(rn_rf)),  rn_rf,  'C1s-', lw=2, label='ResNet-50 (approx)')
ax.plot(range(1, len(dil_rf)), dil_rf[1:], 'C2^-', lw=2, label='Dilated network')
ax.set_xlabel('Layer index'); ax.set_ylabel('Receptive field size (pixels)')
ax.set_title('Receptive Field Growth by Architecture')
ax.legend(); plt.tight_layout(); plt.show()


---
## Exercise 8 ★★☆ — im2col: Conv as Matrix Multiplication

**Task:**
1. Implement `im2col(X, kH, kW, stride, padding)` for a multi-channel input.
2. Implement `col2im(P, X_shape, kH, kW, stride, padding)` (the adjoint, needed for backprop).
3. Verify that `X ≈ col2im(im2col(X))` (modulo overlap averaging).
4. Benchmark: compare time for loop-based conv vs im2col + `np.dot`.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def im2col(X, kH, kW, stride=1, padding=0):
        """
        X: (C, H, W)
        Returns P: (C*kH*kW, H_out*W_out)
        """
        pass  # TODO

    def col2im(P, X_shape, kH, kW, stride=1, padding=0):
        """
        P: (C*kH*kW, H_out*W_out)
        X_shape: (C, H, W)
        Returns X_approx: (C, H, W)
        """
        pass  # TODO

    X_t = np.random.randn(3, 8, 8)
    P_t = im2col(X_t, 3, 3, padding=1)
    print("im2col shape:", P_t.shape if P_t is not None else None)


### Solution 8

In [ ]:
import time

def im2col(X, kH, kW, stride=1, padding=0):
    C, H, W = X.shape
    if padding:
        X = np.pad(X, ((0,0),(padding,padding),(padding,padding)))
    _, Hp, Wp = X.shape
    H_out = (Hp - kH) // stride + 1
    W_out = (Wp - kW) // stride + 1
    cols = []
    for i in range(0, Hp - kH + 1, stride):
        for j in range(0, Wp - kW + 1, stride):
            cols.append(X[:, i:i+kH, j:j+kW].ravel())
    return np.array(cols).T   # (C*kH*kW, H_out*W_out)

def col2im(P, X_shape, kH, kW, stride=1, padding=0):
    C, H, W = X_shape
    Hp = H + 2*padding
    Wp = W + 2*padding
    H_out = (Hp - kH) // stride + 1
    W_out = (Wp - kW) // stride + 1
    X_pad = np.zeros((C, Hp, Wp))
    col_idx = 0
    for i in range(H_out):
        for j in range(W_out):
            X_pad[:, i*stride:i*stride+kH, j*stride:j*stride+kW] +=                 P[:, col_idx].reshape(C, kH, kW)
            col_idx += 1
    if padding:
        return X_pad[:, padding:-padding, padding:-padding]
    return X_pad

# Verify im2col → col2im round-trip (with overlap averaging is approximate)
X_t = np.arange(48, dtype=float).reshape(3, 4, 4)
P_t = im2col(X_t, 2, 2, stride=2, padding=0)
X_back = col2im(P_t, X_t.shape, 2, 2, stride=2, padding=0)
print("im2col shape:", P_t.shape)
print("col2im shape:", X_back.shape)
print("Round-trip (stride=2, no overlap): exact?",
      np.allclose(X_t, X_back))  # should be exact for non-overlapping

# Benchmark: loop conv vs GEMM
import time
C_in, H, W, C_out, kH, kW = 16, 32, 32, 32, 3, 3
X_bench = np.random.randn(C_in, H, W)
W_bench = np.random.randn(C_out, C_in * kH * kW)

# Loop conv (simplified: use im2col internally for fairness of computation)
reps = 5

t0 = time.perf_counter()
for _ in range(reps):
    P = im2col(X_bench, kH, kW, padding=1)
    Y_gemm = W_bench @ P
t_gemm = (time.perf_counter()-t0)/reps * 1000

print(f"\nim2col + GEMM: {t_gemm:.2f} ms")
print(f"Output shape: {Y_gemm.shape}  (C_out × H_out*W_out)")
print(f"\nim2col matrix size: {P.shape}  = {P.nbytes/1024:.1f} KB")
print(f"Input size: {X_bench.nbytes/1024:.1f} KB")
print(f"Memory overhead: {P.nbytes / X_bench.nbytes:.1f}x")


---
## Exercise 9 ★★☆ — ResNet Residual Block Implementation

**Task:**
1. Implement a residual block (basic and bottleneck variants) using NumPy.
2. Show mathematically and empirically why skip connections enable gradient flow.
3. Count parameters for both variants and compare.
4. Verify the "gradient highway" property: with zero-initialised residual branch,
   gradients pass through unchanged.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    class ReLU:
        def forward(self, x):
            self.mask = x > 0
            return x * self.mask
        def backward(self, dx):
            return dx * self.mask

    class ResidualBlock:
        """Basic 2-layer residual block (simplified, 1-D vectors for clarity)."""
        def __init__(self, dim):
            self.W1 = np.random.randn(dim, dim) * 0.01
            self.W2 = np.random.randn(dim, dim) * 0.01
            self.relu = ReLU()
            self._cache = {}

        def forward(self, x):
            pass  # TODO: implement y = F(x) + x

        def backward(self, dy):
            pass  # TODO: compute dx, dW1, dW2

    block = ResidualBlock(4)
    x_t   = np.random.randn(4)
    y_t   = block.forward(x_t)
    print("Output shape:", y_t.shape if y_t is not None else None)


### Solution 9

In [ ]:
class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, dx):
        return dx * self.mask

class ResidualBlock:
    """
    Basic residual block: y = F(x; W1, W2) + x
    where F = ReLU(W2 @ ReLU(W1 @ x))
    """
    def __init__(self, dim, zero_init=False):
        scale = 0.0 if zero_init else 0.1
        self.W1 = np.random.randn(dim, dim) * scale
        self.W2 = np.random.randn(dim, dim) * scale
        self.relu1 = ReLU()
        self.relu2 = ReLU()
        self._cache = {}

    def forward(self, x):
        h1 = self.relu1.forward(self.W1 @ x)   # first conv/FC
        h2 = self.relu2.forward(self.W2 @ h1)  # second conv/FC
        y  = h2 + x                             # skip connection
        self._cache = {'x': x, 'h1': h1, 'h2': h2}
        return y

    def backward(self, dy):
        x, h1, h2 = (self._cache[k] for k in ['x','h1','h2'])
        # Skip connection: gradient splits into two paths
        # dy goes through both F-branch and identity branch
        dF  = dy               # gradient through skip (identity)
        dxI = dy               # gradient through skip

        # Through F branch: dy → relu2 → W2 → relu1 → W1
        dh2 = self.relu2.backward(dF)
        dW2 = np.outer(dh2, h1)
        dh1 = self.W2.T @ dh2
        dh1 = self.relu1.backward(dh1)
        dW1 = np.outer(dh1, x)
        dxF = self.W1.T @ dh1

        # Total dx = gradient through identity + gradient through F
        dx = dxI + dxF
        return dx, dW1, dW2

# ── Gradient highway test ───────────────────────────────────────────────────
dim = 8
# Zero-initialised residual branch → F(x) ≈ 0 → y ≈ x
block_zero = ResidualBlock(dim, zero_init=True)
x_t = np.random.randn(dim)
y_t = block_zero.forward(x_t)
print("Zero-init: F(x) ≈ 0?", np.allclose(y_t, x_t, atol=1e-10))

dy = np.random.randn(dim)
dx, _, _ = block_zero.backward(dy)
print("Zero-init: dx ≈ dy (gradient passes through)?",
      np.allclose(dx, 2*dy, atol=1e-10))  # dy + dxF ≈ dy + dy*(W1~0)≈dy

# With normal init — gradient still has clean path
block_norm = ResidualBlock(dim, zero_init=False)
y_n = block_norm.forward(x_t)
dx_n, _, _ = block_norm.backward(dy)
# Even with random weights, at least dy passes through unchanged
print(f"Normal-init: |dx - dy| = {np.linalg.norm(dx_n - dy):.4f}  (extra from F branch)")

# Parameter comparison
def basic_params(C):
    return 2 * C * C  # two C×C weight matrices

def bottleneck_params(C, expansion=4):
    mid = C // expansion
    return C*mid + mid*mid*9 + mid*C   # 1×1 + 3×3 + 1×1 (approx for 2-D conv)

dims = [64, 128, 256, 512]
print("\nC      Basic      Bottleneck   Savings")
for C in dims:
    b = basic_params(C)
    bn = bottleneck_params(C)
    print(f"{C:4d}  {b:8,d}  {bn:10,d}  {1-bn/b:.1%}")

# Visualise gradient flow
depths = np.arange(1, 51)
resnet_grad = (1 + 0.3)**depths          # residual: (1 + ∂F/∂x) per layer
plain_grad  = 0.9**depths                # plain: can vanish or explode
plain_exp   = 1.1**depths

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(depths, resnet_grad, 'C0-', lw=2, label='ResNet (1 + 0.3 per layer)')
ax.semilogy(depths, plain_grad,  'C1--', lw=2, label='Plain (0.9 per layer, vanishing)')
ax.semilogy(depths, plain_exp,   'C2:', lw=2, label='Plain (1.1 per layer, exploding)')
ax.axhline(1.0, color='gray', linestyle=':', lw=1)
ax.set_xlabel('Network depth'); ax.set_ylabel('Gradient magnitude (log scale)')
ax.set_title('Gradient Flow: ResNet vs Plain Networks')
ax.legend(); plt.tight_layout(); plt.show()


---
## Exercise 10 ★★★ — Kernel Separability via SVD

A 2-D filter $\mathbf{K}\in\mathbb{R}^{k\times k}$ is *separable* iff $\text{rank}(\mathbf{K})=1$,
i.e. $\mathbf{K} = \mathbf{v}\mathbf{h}^T$ for column vectors $\mathbf{v}, \mathbf{h}$.

This can be detected via SVD: $\mathbf{K} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$.
If $\sigma_1 \gg \sigma_2 \approx 0$, the kernel is (approximately) separable.

**Task:**
1. Generate several kernels: Sobel, Gaussian, random, and a low-rank approximation.
2. Compute SVD and check separability criterion.
3. For separable kernels, extract $\mathbf{v}$ and $\mathbf{h}$ and verify $\mathbf{v}\mathbf{h}^T \approx \mathbf{K}$.
4. Show computational savings: $O(k^2 N)$ for full conv vs $O(2kN)$ for separable.


In [ ]:
# Scaffold preserved for interactive work.
if False:
    # ── YOUR SOLUTION HERE ────────────────────────────────────────────────────────

    def check_separability(K, tol=1e-6):
        """
        Returns (is_separable, v, h, singular_values)
        where v@h.T ≈ K if separable.
        """
        pass  # TODO

    kernels = {
        'Sobel-x': np.array([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=float),
        'Gaussian-3': None,  # TODO: generate Gaussian
        'Random-3':   np.random.randn(3, 3),
        'Random-5':   np.random.randn(5, 5),
    }
    for name, K in kernels.items():
        if K is None: continue
        sep, v, h, sv = check_separability(K)
        print(f"{name:15s}: separable={sep}  σ=[{', '.join(f'{s:.3f}' for s in sv[:3])}]")


### Solution 10

In [ ]:
def make_gaussian_kernel(k, sigma=1.0):
    """Isotropic Gaussian kernel of size k×k."""
    coords = np.arange(k) - k//2
    g1d = np.exp(-coords**2 / (2*sigma**2))
    g1d /= g1d.sum()
    return np.outer(g1d, g1d)

def check_separability(K, tol=1e-8):
    U, sv, Vt = np.linalg.svd(K)
    # Check if all but first singular value are negligible
    is_sep = all(s < tol * sv[0] for s in sv[1:])
    if is_sep or sv[1] < 1e-2 * sv[0]:  # also catch near-separable
        v = U[:, 0] * np.sqrt(sv[0])
        h = Vt[0, :] * np.sqrt(sv[0])
    else:
        v = h = None
    return is_sep, v, h, sv

kernels = {
    'Sobel-x':    np.array([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=float),
    'Sobel-y':    np.array([[1,2,1],[0,0,0],[-1,-2,-1]], dtype=float),
    'Gaussian-3': make_gaussian_kernel(3, sigma=1.0),
    'Gaussian-5': make_gaussian_kernel(5, sigma=1.5),
    'Random-3':   np.random.randn(3, 3),
    'Random-5':   np.random.randn(5, 5),
    'Laplacian':  np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=float),
}

print(f"{'Kernel':<14} {'Separable':>10} {'σ₁':>8} {'σ₂':>8} {'σ₂/σ₁':>8}")
print("-"*52)
for name, K in kernels.items():
    sep, v, h, sv = check_separability(K)
    ratio = sv[1]/sv[0] if len(sv) > 1 else 0.0
    sep_str = "✓ YES" if sep else "✗ NO"
    print(f"{name:<14} {sep_str:>10} {sv[0]:>8.3f} {sv[1] if len(sv)>1 else 0:>8.4f} {ratio:>8.4f}")

# Verify: Gaussian is separable → v ⊗ h^T ≈ K
K_gauss = kernels['Gaussian-5']
_, v_sep, h_sep, _ = check_separability(K_gauss)
K_approx = np.outer(v_sep, h_sep)
print(f"\nGaussian-5 separable decomp error: {np.max(np.abs(K_gauss - K_approx)):.2e}")

# Visualise SVD decomposition
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
test_kernels = ['Sobel-x', 'Gaussian-5', 'Random-5', 'Laplacian']
for col, name in enumerate(test_kernels):
    K = kernels[name]
    _, sv_k, _ = np.linalg.svd(K)
    im = axes[0][col].imshow(K, cmap='RdBu', aspect='auto')
    axes[0][col].set_title(name); plt.colorbar(im, ax=axes[0][col])
    axes[1][col].bar(range(len(sv_k)), sv_k, color='C0', alpha=0.8)
    axes[1][col].set_title(f'Singular values')
    axes[1][col].set_xlabel('Index')
plt.suptitle('Kernel Separability: Singular Value Decomposition', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Computational savings
print("\nComputational savings from separable convolution:")
print(f"{'k':>3} {'Full ops/px':>14} {'Sep. ops/px':>14} {'Savings':>10}")
print("-"*45)
for k in [3, 5, 7, 11, 15]:
    full = k * k
    sep  = 2 * k
    print(f"{k:3d} {full:14d} {sep:14d} {1-sep/full:10.1%}")


---
## Completion

You have worked through:

| Exercise | Topic | Difficulty |
|---|---|---|
| 1 | Output size formula with all parameters | ★☆☆ |
| 2 | Parameter counting for all layer types | ★☆☆ |
| 3 | Full 2-D conv implementation (stride, padding, dilation) | ★★☆ |
| 4 | Sobel edge detection and separability | ★☆☆ |
| 5 | Conv backprop with numerical gradient verification | ★★☆ |
| 6 | BatchNorm full backward pass derivation | ★★★ |
| 7 | Receptive field computation across architectures | ★★☆ |
| 8 | im2col / col2im for GEMM-based conv | ★★☆ |
| 9 | ResNet residual block and gradient highway | ★★☆ |
| 10 | Kernel separability via SVD | ★★★ |

**Key takeaways:**
- Convolution = structured sparse linear map (Toeplitz/BCCS matrix)
- Backprop through conv = another convolution (with flipped kernel or transposed)
- BatchNorm backward requires careful chain-rule through shared statistics
- Separable kernels (Gaussian, Sobel) give free $O(k)$ speedup
- Skip connections make gradient flow = gradient highway, preventing vanishing gradients
- ViT patch embedding = strided Conv2d — same math, different inductive biases
